In [1]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import math

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
content = path.read_bytes()

expected = json.loads(
    path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(content).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(content)
group_rows = []

for layer in plan["inventory"]:
    out_channels, in_channels, kh, kw = layer["weight_shape"]
    if math.prod(layer["weight_shape"]) != layer["weight_count"]:
        raise RuntimeError("Weight count mismatch.")

    if (kh, kw) == (3, 3):
        groups = out_channels * in_channels
        remainder = 0
    elif (kh, kw) == (1, 1):
        # Keep groups inside each output channel.
        groups = out_channels * (in_channels // 9)
        remainder = out_channels * (in_channels % 9)
    else:
        raise RuntimeError("Unsupported kernel shape.")

    grouped = groups * 9
    if grouped + remainder != layer["weight_count"]:
        raise RuntimeError("Grouping accounting mismatch.")

    group_rows.append({
        "layer": layer["name"],
        "groups": groups,
        "grouped_weights": grouped,
        "ungrouped_weights": remainder,
    })

total = sum(layer["weight_count"] for layer in plan["inventory"])
groups = sum(row["groups"] for row in group_rows)
grouped = sum(row["grouped_weights"] for row in group_rows)
remainder = sum(row["ungrouped_weights"] for row in group_rows)

print("Eligible layers:", len(group_rows))
print("Total logical INT8 weights:", total)
print("Nine-weight groups:", groups)
print("Grouped weights:", grouped)
print("Ungrouped tail weights:", remainder)
print("Grouped weight fraction:", grouped / total)

# Two three-bit masks per group: force-zero and force-one.
print("Ideal six-bit-mask storage, bytes:", (6 * groups + 7) // 8)
print("One-byte-per-group mask storage, bytes:", groups)
print("One-byte mask overhead / logical INT8 weights, %:", 100 * groups / total)
print("These estimates exclude headers and other metadata.")

print("\nLayers with ungrouped tails:")
for row in group_rows:
    if row["ungrouped_weights"]:
        print(row["layer"], "| tail weights:", row["ungrouped_weights"])

print("\nActual invariant-bit coverage has not been measured.")
print("No model loading, inference, or project file changes.")

Mounted at /content/drive
Eligible layers: 63
Total logical INT8 weights: 11146080
Nine-weight groups: 1235584
Grouped weights: 11120256
Ungrouped tail weights: 25824
Grouped weight fraction: 0.9976831316480772
Ideal six-bit-mask storage, bytes: 926688
One-byte-per-group mask storage, bytes: 1235584
One-byte mask overhead / logical INT8 weights, %: 11.08536812942308
These estimates exclude headers and other metadata.

Layers with ungrouped tails:
model.2.cv1.conv | tail weights: 64
model.2.cv2.conv | tail weights: 384
model.4.cv1.conv | tail weights: 256
model.4.cv2.conv | tail weights: 512
model.6.cv1.conv | tail weights: 1024
model.6.cv2.conv | tail weights: 2048
model.8.cv1.conv | tail weights: 4096
model.8.cv2.conv | tail weights: 1536
model.9.cv1.conv | tail weights: 2048
model.9.cv2.conv | tail weights: 3584
model.12.cv1.conv | tail weights: 768
model.12.cv2.conv | tail weights: 1536
model.15.cv1.conv | tail weights: 768
model.15.cv2.conv | tail weights: 384
model.18.cv1.conv | t

In [2]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
plan_path = ROOT / "experiments/notebook05b/int8_screening_plan_v1.json"
plan_bytes = plan_path.read_bytes()

expected = json.loads(
    plan_path.with_suffix(".sha256.json").read_bytes()
)["sha256"]
if hashlib.sha256(plan_bytes).hexdigest() != expected:
    raise RuntimeError("Screening plan checksum mismatch.")

plan = json.loads(plan_bytes)
protocol = {
    "version": "1.0",
    "status": "FROZEN_BASELINE_BEFORE_INVARIANT_MASK_EXTRACTION",
    "reference_doi": "10.1007/s11227-024-06693-7",
    "implementation_claim": "documented_adaptation_not_exact_paper_reproduction",
    "representation": "05a_shared_logical_INT8_per_output_channel",
    "checkpoint_sha256": plan["checkpoint_sha256"],
    "clean_model_state_sha256": plan["clean_model_state_sha256"],
    "source_screening_plan_sha256": expected,
    "scope": "63_eligible_Conv2d_weight_tensors",
    "protected_candidate_bits": [4, 5, 6],
    "grouping": {
        "3x3": "one_9_weight_kernel_per_output_input_channel_pair",
        "1x1": "consecutive_groups_of_9_input_weights_within_each_output_channel",
        "order": "contiguous_OIHW_tensor_order",
        "tails": "leave_fewer_than_9_remaining_weights_unprotected",
        "cross_output_channel_grouping": False
    },
    "mask_definition": {
        "force_zero": "candidate_bit_is_zero_in_all_9_clean_weights",
        "force_one": "candidate_bit_is_one_in_all_9_clean_weights",
        "other_bits": "unprotected_and_unchanged",
        "zero_one_masks_must_be_disjoint": True,
        "storage": "one_uint8_per_group",
        "packing": (
            "bits_0_1_2_force_zero_for_weight_bits_4_5_6; "
            "bits_3_4_5_force_one_for_weight_bits_4_5_6; "
            "bits_6_7_reserved_zero"
        )
    },
    "operation": {
        "reference": "extract_once_from_verified_clean_integer_weights",
        "runtime": "scan_all_weights_in_the_selected_protected_groups",
        "correction": "set_only_protected_bits_to_the_stored_reference_value",
        "fault_location_provided_to_method": False,
        "clean_target_values_available_at_runtime": False
    },
    "trust_assumptions": [
        "reference_masks_and_group_metadata_are_trusted",
        "quantization_scales_are_trusted",
        "checker_and_correction_computation_are_trusted"
    ],
    "initial_checks": [
        "clean_weights_are_unchanged",
        "every_protected_single_bit_flip_is_exactly_restored",
        "unprotected_single_bit_flips_are_left_unchanged",
        "tails_are_unchanged",
        "mask_encoding_and_decoding_are_consistent",
        "signed_INT8_boundaries_are_handled_correctly"
    ],
    "coverage_reporting": {
        "grouped_weight_fraction": "not_fault_protection_coverage",
        "candidate_bit_coverage": "protected_weight_bit_positions_over_all_weight_positions_in_bits_4_5_6",
        "all_bit_coverage": "protected_weight_bit_positions_over_8_times_all_eligible_weights",
        "report_per_layer": True,
        "include_unprotected_tails_in_denominators": True
    },
    "cost_reporting": [
        "actual_mask_array_bytes",
        "actual_serialized_metadata_and_file_bytes",
        "ideal_6_bit_packed_size_as_theoretical_reference_only",
        "mask_construction_time",
        "scan_and_correction_time"
    ],
    "limitations": [
        "Not a new protection mechanism claim.",
        "No hardware implementation or native_INT8_inference claim.",
        "No AP improvement established by bit coverage alone.",
        "Trusted metadata faults and activation faults are outside scope.",
        "Protection allocation comparisons require a separate frozen protocol."
    ],
    "existing_05b_and_06b_protocols_changed": False
}

path = ROOT / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json"
content = (
    json.dumps(protocol, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

path.parent.mkdir(parents=True, exist_ok=True)
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing protocol differs; nothing overwritten.")
    print("IDENTICAL PROTOCOL ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW PROTOCOL SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No mask extraction, model loading, or inference.")

NEW PROTOCOL SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/experiments/notebook07a/invariant_bit_baseline_protocol_v1.json
SHA256: 74785069999784731bcc2917a5fd44fa8b698c102f76a9043b1f022b8303a799
No mask extraction, model loading, or inference.


In [3]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError

protected = ("torch", "numpy", "scikit-learn")
before = {name: version(name) for name in protected}

try:
    installed = version("ultralytics")
except PackageNotFoundError:
    installed = None

if installed is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-deps", "ultralytics==8.4.189",
        ],
        check=True,
    )
elif installed != "8.4.189":
    raise RuntimeError(
        f"Unexpected Ultralytics version: {installed}; nothing changed."
    )

after = {name: version(name) for name in protected}
if before != after:
    raise RuntimeError("Protected package versions changed.")

from ultralytics import YOLO
import torch

print("Ultralytics:", version("ultralytics"))
print("Protected versions unchanged:", after)
print("CUDA available:", torch.cuda.is_available())
print("YOLO import: PASS")
print("No checkpoint loading, mask extraction, or inference.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics: 8.4.189
Protected versions unchanged: {'torch': '2.14.1+cpu', 'numpy': '2.4.6', 'scikit-learn': '1.6.1'}
CUDA available: False
YOLO import: PASS
No checkpoint loading, mask extraction, or inference.


In [4]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_extract_v1.py"

source = r'''"""Extract frozen invariant-bit masks on CPU; no COCO image inference."""
import hashlib
import importlib.util
import io
import json
import os
import platform
import time
from pathlib import Path

PROTOCOL_SHA = "74785069999784731bcc2917a5fd44fa8b698c102f76a9043b1f022b8303a799"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def publish(path, content):
    partial = path.with_name(path.name + ".part")
    if path.exists() or partial.exists():
        raise FileExistsError(f"Existing output requires review: {path.name}")
    with partial.open("xb") as stream:
        stream.write(content)
        stream.flush()
        os.fsync(stream.fileno())
    if partial.read_bytes() != content:
        raise RuntimeError("Write verification failed.")
    partial.rename(path)

def run(repo):
    import numpy as np
    import torch
    from importlib.metadata import version

    repo = Path(repo)
    out = repo / "results/notebook07a/invariant_masks_v1"
    if out.exists():
        raise FileExistsError("Output directory exists; inspect before rerunning.")

    protocol = json.loads(verified(
        repo / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json",
        PROTOCOL_SHA
    ))
    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    screen_path = repo / "src/screen05b_campaign_v1.py"
    verified(screen_path, plan["source_sha256"])
    spec = importlib.util.spec_from_file_location("screen07a_verified", screen_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != protocol["clean_model_state_sha256"]:
        raise RuntimeError("Shared clean model mismatch.")

    started = time.perf_counter()
    masks, inventory = {}, []
    popcount = np.array([int(i).bit_count() for i in range(256)], dtype=np.uint8)

    for index, item in enumerate(plan["inventory"]):
        name = item["name"]
        q = prepared.quantized[name][0].cpu().contiguous().numpy()
        if list(q.shape) != item["weight_shape"] or str(q.dtype) != "int8":
            raise RuntimeError("Quantized tensor shape or dtype mismatch.")
        o, i, kh, kw = q.shape
        unsigned = q.view(np.uint8)

        if (kh, kw) == (3, 3):
            groups = unsigned.reshape(-1, 9)
            tail = 0
        elif (kh, kw) == (1, 1):
            complete_inputs = (i // 9) * 9
            groups = unsigned.reshape(o, i)[:, :complete_inputs].reshape(-1, 9)
            tail = o * (i - complete_inputs)
        else:
            raise RuntimeError("Unsupported kernel shape.")

        all_one = np.bitwise_and.reduce(groups, axis=1) & 0x70
        all_zero = np.bitwise_not(
            np.bitwise_or.reduce(groups, axis=1)
        ) & 0x70
        if np.any(all_one & all_zero):
            raise RuntimeError("Conflicting masks.")

        packed = (
            (all_zero >> 4) | ((all_one >> 4) << 3)
        ).astype(np.uint8)
        if np.any(packed & 0xC0):
            raise RuntimeError("Reserved mask bits are not zero.")

        key = f"layer_{index:03d}"
        masks[key] = packed
        protected = int(popcount[packed].sum()) * 9
        count = int(q.size)
        if len(groups) * 9 + tail != count:
            raise RuntimeError("Grouping count mismatch.")

        inventory.append({
            "layer_id": index,
            "layer": name,
            "shape": list(q.shape),
            "array_key": key,
            "clean_integer_array_sha256": sha(q.tobytes()),
            "mask_array_sha256": sha(packed.tobytes()),
            "weights": count,
            "groups": len(groups),
            "tail_weights": tail,
            "mask_array_bytes": packed.nbytes,
            "protected_weight_bit_positions": protected,
            "candidate_bit_coverage": protected / (3 * count),
            "all_bit_coverage": protected / (8 * count),
            "protected_positions_by_bit": {
                str(bit): int(np.count_nonzero(
                    (all_zero | all_one) & (1 << bit)
                )) * 9
                for bit in (4, 5, 6)
            }
        })

    prepared._assert_state(prepared.clean_hash)
    extraction_seconds = time.perf_counter() - started
    if len(inventory) != 63:
        raise RuntimeError("Expected 63 layers.")

    buffer = io.BytesIO()
    np.savez_compressed(buffer, **masks)
    archive_bytes = buffer.getvalue()
    with np.load(io.BytesIO(archive_bytes), allow_pickle=False) as archive:
        if set(archive.files) != set(masks):
            raise RuntimeError("Mask archive keys mismatch.")
        for key, array in masks.items():
            if not np.array_equal(array, archive[key]):
                raise RuntimeError("Mask archive round-trip mismatch.")

    total = sum(row["weights"] for row in inventory)
    groups = sum(row["groups"] for row in inventory)
    protected = sum(row["protected_weight_bit_positions"] for row in inventory)
    payload = {
        "status": "MASKS_EXTRACTED_CORRECTION_TESTS_PENDING",
        "protocol_sha256": PROTOCOL_SHA,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "source_screen_sha256": plan["source_sha256"],
        "checkpoint_sha256": protocol["checkpoint_sha256"],
        "clean_model_state_sha256": prepared.clean_hash,
        "mask_archive_sha256": sha(archive_bytes),
        "environment": {
            "python": platform.python_version(),
            "packages": {p: version(p) for p in ("torch", "numpy", "ultralytics")},
            "device": "cpu",
            "torch_threads": torch.get_num_threads()
        },
        "totals": {
            "weights": total,
            "groups": groups,
            "tail_weights": sum(row["tail_weights"] for row in inventory),
            "mask_array_bytes": sum(array.nbytes for array in masks.values()),
            "ideal_six_bit_storage_bytes": (6 * groups + 7) // 8,
            "compressed_mask_archive_bytes": len(archive_bytes),
            "protected_weight_bit_positions": protected,
            "candidate_bit_coverage": protected / (3 * total),
            "all_bit_coverage": protected / (8 * total),
            "extraction_seconds_excluding_model_preparation": extraction_seconds
        },
        "inventory": inventory,
        "limitations": [
            "Trusted masks, metadata and scales.",
            "Actual protection coverage differs from grouped weight fraction.",
            "Compressed file size is not runtime mask memory.",
            "Correction correctness and AP effectiveness are not yet tested."
        ]
    }
    manifest_bytes = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()

    out.mkdir(parents=True, exist_ok=False)
    publish(out / "masks.npz", archive_bytes)
    publish(out / "manifest.json", manifest_bytes)

    print(json.dumps(payload["totals"], indent=2))
    print("Serialized manifest bytes:", len(manifest_bytes))
    print("Total archive + manifest bytes:", len(archive_bytes) + len(manifest_bytes))
    print("Saved:", out)
    print("Manifest SHA256:", sha(manifest_bytes))
    print("No COCO images, correction experiment, or 05b execution.")
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Extraction has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/invariant07a_extract_v1.py
Syntax: PASS
Extraction has not been executed.


In [5]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_extract_v1.py"

spec = importlib.util.spec_from_file_location("invariant07a_extract_v1", path)
invariant07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(invariant07a)

mask_manifest07a = invariant07a.run(ROOT)

{
  "weights": 11146080,
  "groups": 1235584,
  "tail_weights": 25824,
  "mask_array_bytes": 1235584,
  "ideal_six_bit_storage_bytes": 926688,
  "compressed_mask_archive_bytes": 147693,
  "protected_weight_bit_positions": 1297125,
  "candidate_bit_coverage": 0.03879166487231386,
  "all_bit_coverage": 0.014546874327117695,
  "extraction_seconds_excluding_model_preparation": 0.28175375000000713
}
Serialized manifest bytes: 50418
Total archive + manifest bytes: 198111
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1
Manifest SHA256: bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a
No COCO images, correction experiment, or 05b execution.


In [6]:
from pathlib import Path
import hashlib
import io
import json
import numpy as np

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
FOLDER = ROOT / "results/notebook07a/invariant_masks_v1"

manifest_bytes = (FOLDER / "manifest.json").read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest() != (
    "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
):
    raise RuntimeError("Manifest checksum mismatch.")

envelope = json.loads(manifest_bytes)
manifest = envelope["payload"]
canonical = json.dumps(
    manifest, sort_keys=True, ensure_ascii=False,
    allow_nan=False, separators=(",", ":")
).encode("utf-8")
if hashlib.sha256(canonical).hexdigest() != envelope["payload_sha256"]:
    raise RuntimeError("Manifest payload mismatch.")

raw = (FOLDER / "masks.npz").read_bytes()
if hashlib.sha256(raw).hexdigest() != manifest["mask_archive_sha256"]:
    raise RuntimeError("Mask archive checksum mismatch.")

totals_by_bit = {bit: 0 for bit in (4, 5, 6)}
total_groups = 0
groups_with_protection = 0

with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
    expected_keys = {row["array_key"] for row in manifest["inventory"]}
    if set(archive.files) != expected_keys:
        raise RuntimeError("Unexpected mask arrays.")

    for row in manifest["inventory"]:
        packed = archive[row["array_key"]]
        if packed.dtype != np.uint8 or packed.shape != (row["groups"],):
            raise RuntimeError("Mask shape or dtype mismatch.")
        if hashlib.sha256(packed.tobytes()).hexdigest() != row["mask_array_sha256"]:
            raise RuntimeError("Layer mask checksum mismatch.")

        zero = packed & 0b111
        one = (packed >> 3) & 0b111

        if np.any(packed & 0b11000000) or np.any(zero & one):
            raise RuntimeError("Invalid or conflicting mask encoding.")
        if not np.array_equal(packed, zero | (one << 3)):
            raise RuntimeError("Mask round-trip mismatch.")

        protected = zero | one
        total_groups += len(packed)
        groups_with_protection += int(np.count_nonzero(protected))

        for offset, bit in enumerate((4, 5, 6)):
            positions = int(np.count_nonzero(protected & (1 << offset))) * 9
            if positions != row["protected_positions_by_bit"][str(bit)]:
                raise RuntimeError("Per-bit coverage mismatch.")
            totals_by_bit[bit] += positions

if sum(totals_by_bit.values()) != manifest["totals"]["protected_weight_bit_positions"]:
    raise RuntimeError("Total coverage mismatch.")

weights = manifest["totals"]["weights"]
print("Mask integrity and encoding: PASS")
print("Groups with at least one protected bit:", groups_with_protection)
print("Total groups:", total_groups)

for bit, count in totals_by_bit.items():
    print(
        f"Bit {bit}: protected weight positions={count:,} | "
        f"coverage within this bit={100 * count / weights:.6f}%"
    )

print("No weight correction or COCO inference executed.")

Mask integrity and encoding: PASS
Groups with at least one protected bit: 100382
Total groups: 1235584
Bit 4: protected weight positions=117,081 | coverage within this bit=1.050423%
Bit 5: protected weight positions=334,899 | coverage within this bit=3.004635%
Bit 6: protected weight positions=845,145 | coverage within this bit=7.582442%
No weight correction or COCO inference executed.


In [7]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_check_v1.py"

source = r'''"""Check invariant-mask correction on CPU; no COCO image inference."""
import hashlib
import importlib.util
import io
import json
import time
from pathlib import Path

MANIFEST_SHA = "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def grouped(values):
    o, i, kh, kw = values.shape
    if (kh, kw) == (3, 3):
        return values.reshape(-1, 9)
    if (kh, kw) == (1, 1):
        end = (i // 9) * 9
        return values.reshape(o, i)[:, :end].reshape(-1, 9)
    raise ValueError("Unsupported kernel shape.")

def correct(observed, packed):
    """Use observed weights and trusted masks only; no clean-weight input."""
    import numpy as np
    if observed.dtype != np.int8 or packed.dtype != np.uint8:
        raise ValueError("Expected int8 weights and uint8 masks.")
    result = observed.copy()
    unsigned = result.view(np.uint8)
    blocks = grouped(unsigned)
    if packed.shape != (len(blocks),):
        raise ValueError("Mask count mismatch.")
    zero = ((packed & 7) << 4).astype(np.uint8)
    one = (((packed >> 3) & 7) << 4).astype(np.uint8)
    if np.any(zero & one) or np.any(packed & 0xC0):
        raise ValueError("Invalid masks.")
    fixed = (blocks & np.bitwise_not(zero[:, None])) | one[:, None]
    o, i, kh, kw = result.shape
    if (kh, kw) == (3, 3):
        unsigned.reshape(-1, 9)[:] = fixed
    else:
        end = (i // 9) * 9
        unsigned.reshape(o, i)[:, :end] = fixed.reshape(o, end)
    return result

def run(repo):
    import numpy as np
    import torch
    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    destination = folder / "correctness_v1.json"
    if destination.exists():
        raise FileExistsError("Correctness report exists; inspect before rerunning.")

    envelope = json.loads(verified(folder / "manifest.json", MANIFEST_SHA))
    manifest = envelope["payload"]
    if sha(canonical(manifest)) != envelope["payload_sha256"]:
        raise RuntimeError("Manifest payload mismatch.")
    protocol = json.loads(verified(
        repo / "experiments/notebook07a/invariant_bit_baseline_protocol_v1.json",
        manifest["protocol_sha256"]
    ))
    plan = json.loads(verified(
        repo / "experiments/notebook05b/int8_screening_plan_v1.json",
        protocol["source_screening_plan_sha256"]
    ))
    screen_path = repo / "src/screen05b_campaign_v1.py"
    verified(screen_path, manifest["source_screen_sha256"])
    if plan["source_sha256"] != manifest["source_screen_sha256"]:
        raise RuntimeError("Source provenance mismatch.")
    spec = importlib.util.spec_from_file_location("screen07a_check", screen_path)
    screen = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(screen)

    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != manifest["clean_model_state_sha256"]:
        raise RuntimeError("Clean model mismatch.")

    raw = verified(folder / "masks.npz", manifest["mask_archive_sha256"])
    cases = recovered = unchanged_unprotected = clean_weights = 0
    rows = []
    started = time.perf_counter()

    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            q = prepared.quantized[item["layer"]][0].cpu().contiguous().numpy()
            packed = archive[item["array_key"]].copy()
            if sha(q.tobytes()) != item["clean_integer_array_sha256"]:
                raise RuntimeError("Clean integer tensor mismatch.")
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Layer mask checksum mismatch.")

            # Independently reconstruct invariant flags from all nine weights.
            blocks = grouped(q.view(np.uint8))
            expected = np.zeros(len(blocks), dtype=np.uint8)
            for offset, bit in enumerate((4, 5, 6)):
                values = (blocks >> bit) & 1
                expected |= np.all(values == 0, axis=1).astype(np.uint8) << offset
                expected |= np.all(values == 1, axis=1).astype(np.uint8) << (offset + 3)
            if not np.array_equal(packed, expected):
                raise RuntimeError("Masks do not match clean group invariants.")

            if not np.array_equal(correct(q, packed), q):
                raise RuntimeError("Clean weights were changed.")
            clean_weights += q.size
            layer_recovered = 0

            for bit in range(8):
                # Batch independent position-wise flips under fixed masks.
                faulty = (q.view(np.uint8) ^ np.uint8(1 << bit)).view(np.int8)
                final = correct(faulty, packed)
                protected_positions = np.zeros(q.shape, dtype=bool)

                if bit in (4, 5, 6):
                    offset = bit - 4
                    flags = (((packed & 7) | ((packed >> 3) & 7)) >> offset) & 1
                    expanded = np.repeat(flags.astype(bool), 9)
                    o, i, kh, kw = q.shape
                    if (kh, kw) == (3, 3):
                        protected_positions.reshape(-1)[:] = expanded
                    else:
                        end = (i // 9) * 9
                        protected_positions.reshape(o, i)[:, :end] = expanded.reshape(o, end)

                if not np.array_equal(final[protected_positions], q[protected_positions]):
                    raise RuntimeError("Protected flip was not restored.")
                if not np.array_equal(final[~protected_positions], faulty[~protected_positions]):
                    raise RuntimeError("Unprotected flip or tail was altered.")

                count = int(protected_positions.sum())
                cases += q.size
                recovered += count
                layer_recovered += count
                unchanged_unprotected += q.size - count

            if layer_recovered != item["protected_weight_bit_positions"]:
                raise RuntimeError("Layer recovery count mismatch.")
            rows.append({
                "layer": item["layer"],
                "position_bit_cases": int(q.size * 8),
                "exactly_recovered_protected_cases": layer_recovered
            })

    prepared._assert_state(prepared.clean_hash)
    if recovered != manifest["totals"]["protected_weight_bit_positions"]:
        raise RuntimeError("Total recovery count mismatch.")
    payload = {
        "status": "CORRECTION_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION",
        "manifest_sha256": MANIFEST_SHA,
        "protocol_sha256": manifest["protocol_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "clean_weights_checked_unchanged": int(clean_weights),
        "position_bit_cases_checked": int(cases),
        "protected_cases_exactly_recovered": int(recovered),
        "unprotected_cases_left_unchanged": int(unchanged_unprotected),
        "check_seconds_excluding_model_preparation": time.perf_counter() - started,
        "method": "Vectorized position-wise checks; fixed masks make corrections separable.",
        "limits": [
            "Not COCO inference or an AP benefit measurement.",
            "Trusted masks, scales, metadata and checker.",
            "This does not validate hardware fault handling or scan performance."
        ],
        "layers": rows
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    for key in (
        "status", "clean_weights_checked_unchanged",
        "position_bit_cases_checked", "protected_cases_exactly_recovered",
        "unprotected_cases_left_unchanged",
        "check_seconds_excluding_model_preparation"
    ):
        print(f"{key}: {payload[key]}")
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("invariant07a_check_v1", path)
check07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(check07a)
correctness07a = check07a.run(ROOT)

status: CORRECTION_FUNCTIONAL_CHECKS_PASS_NO_AP_EVALUATION
clean_weights_checked_unchanged: 11146080
position_bit_cases_checked: 89168640
protected_cases_exactly_recovered: 1297125
unprotected_cases_left_unchanged: 87871515
check_seconds_excluding_model_preparation: 1.1109702830000288
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/correctness_v1.json
SHA256: b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206


In [8]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_compare_v1.py"

source = r'''"""Retrospective comparison on existing 06b test targets; no inference."""
import hashlib
import io
import json
from pathlib import Path

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def record(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Record checksum mismatch.")
    return envelope["payload"]

def run(repo):
    import numpy as np
    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    manifest = record(verified(
        folder / "manifest.json",
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
    ))
    correctness = record(verified(
        folder / "correctness_v1.json",
        "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206"
    ))
    if correctness["manifest_sha256"] != (
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
    ):
        raise RuntimeError("Correctness provenance mismatch.")

    source06a = repo / "results/notebook06a"
    session = record(
        (source06a / "masked_bit_comparison_v1/session.json").read_bytes()
    )
    dataset_bytes = verified(
        source06a / "masked_bit_dataset_v1/dataset.npz",
        session["dataset_sha256"]
    )
    with np.load(io.BytesIO(dataset_bytes), allow_pickle=False) as archive:
        test = {key: archive[f"test_{key}"].copy()
                for key in ("bits", "layer", "channel", "flat_index")}

    folder06b = repo / "results/notebook06b/evaluation_v1"
    summary = record((folder06b / "summary.json").read_bytes())
    if sha(canonical(session)) != summary["source_session_fingerprint"]:
        raise RuntimeError("06a/06b session mismatch.")
    outcome_bytes = verified(
        folder06b / "outcomes.npz", summary["outcomes_sha256"]
    )
    with np.load(io.BytesIO(outcome_bytes), allow_pickle=False) as archive:
        outcomes = {key: archive[key].copy() for key in archive.files}
    for key in ("layer", "channel", "flat_index"):
        if not np.array_equal(test[key], outcomes[key]):
            raise RuntimeError("Test target alignment mismatch.")

    n = len(test["bits"])
    recovered = np.zeros((n, 8), dtype=bool)
    mask_bytes = verified(folder / "masks.npz", manifest["mask_archive_sha256"])

    with np.load(io.BytesIO(mask_bytes), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            selected = np.flatnonzero(test["layer"] == item["layer_id"])
            o, i, kh, kw = item["shape"]
            flat = test["flat_index"][selected]
            if np.any(flat < 0) or np.any(flat >= o * i * kh * kw):
                raise RuntimeError("Invalid target index.")
            channel = flat // (i * kh * kw)
            if not np.array_equal(channel, test["channel"][selected]):
                raise RuntimeError("Target channel mismatch.")

            if (kh, kw) == (3, 3):
                valid = np.ones(len(selected), dtype=bool)
                group_index = flat // 9
            elif (kh, kw) == (1, 1):
                within = flat % i
                valid = within < (i // 9) * 9
                group_index = channel * (i // 9) + within // 9
            else:
                raise RuntimeError("Unsupported kernel.")

            packed = archive[item["array_key"]]
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Layer mask mismatch.")
            rows = selected[valid]
            codes = packed[group_index[valid]]
            zero, one = codes & 7, (codes >> 3) & 7

            for offset, bit in enumerate((4, 5, 6)):
                force_zero = ((zero >> offset) & 1).astype(bool)
                force_one = ((one >> offset) & 1).astype(bool)
                clean_bits = test["bits"][rows, bit]
                if np.any(clean_bits[force_zero] != 0) or np.any(clean_bits[force_one] != 1):
                    raise RuntimeError("Mask disagrees with clean sampled targets.")
                recovered[rows, bit] = force_zero | force_one

    methods = {
        "invariant_group_baseline": {
            "exact_recovery_fraction": float(recovered.mean()),
            "exact_recovery_by_bit": recovered.mean(axis=0).tolist(),
            "clean_false_intervention_fraction": 0.0,
            "clean_control_basis": "Full clean-weight functional check and invariant agreement.",
        }
    }
    for method in ("C", "D"):
        values = outcomes[f"{method}_exact_recovery"]
        methods[method] = {
            "exact_recovery_fraction": float(values.mean()),
            "exact_recovery_by_bit": values.mean(axis=0).tolist(),
            "clean_false_intervention_fraction": float(
                outcomes[f"{method}_clean_intervention"].mean()
            )
        }

    payload = {
        "status": "RETROSPECTIVE_COMMON_TARGET_COMPARISON_NOT_EQUAL_BUDGET",
        "targets": n,
        "single_bit_cases_per_method": n * 8,
        "dataset_sha256": session["dataset_sha256"],
        "outcomes_sha256": summary["outcomes_sha256"],
        "mask_archive_sha256": manifest["mask_archive_sha256"],
        "module_sha256": sha(Path(__file__).read_bytes()),
        "methods": methods,
        "limitations": [
            "Existing inspected 06b test targets; exploratory comparison.",
            "Recovery of the invariant baseline derived from verified bit coverage.",
            "Predictor and mask memory budgets are not matched.",
            "Uniformity over sampled targets and bits is not whole-network uniform sampling.",
            "No new COCO inference or AP benefit established."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    destination = folder / "comparison_on_06b_targets_v1.json"
    if destination.exists():
        if destination.read_bytes() != content:
            raise RuntimeError("Existing comparison differs; nothing overwritten.")
    else:
        with destination.open("xb") as stream:
            stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    print("Common test targets:", n)
    print("Single-bit cases per method:", n * 8)
    for method, result in methods.items():
        print(
            f"{method}: exact recovery={100 * result['exact_recovery_fraction']:.6f}% | "
            f"clean false intervention={100 * result['clean_false_intervention_fraction']:.6f}%"
        )
        print("Recovery by bit, %:",
              [round(100 * x, 6) for x in result["exact_recovery_by_bit"]])
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
else:
    with path.open("xb") as stream:
        stream.write(content)

assert path.read_bytes() == content
spec = importlib.util.spec_from_file_location("invariant07a_compare_v1", path)
compare07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(compare07a)
comparison07a = compare07a.run(ROOT)

Common test targets: 16061
Single-bit cases per method: 128488
invariant_group_baseline: exact recovery=1.180655% | clean false intervention=0.000000%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.0, 1.351099, 2.776913, 5.317228, 0.0]
C: exact recovery=17.424195% | clean false intervention=0.958844%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.211693, 4.495361, 15.490941, 48.602204, 70.593363]
D: exact recovery=12.257954% | clean false intervention=0.765830%
Recovery by bit, %: [0.0, 0.0, 0.0, 0.0, 0.0, 1.351099, 23.852811, 72.859722]
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/comparison_on_06b_targets_v1.json
SHA256: 3410026a67e06483f4fff3e6823c9a7dcf47ee6c9c6d658693937983214e9254


In [9]:
from pathlib import Path
import hashlib
import io
import json
import joblib

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
FOLDER = ROOT / "results/notebook06a/masked_bit_comparison_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def read_record(path):
    envelope = json.loads(path.read_bytes())
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {path.name}")
    return payload

session = read_record(FOLDER / "session.json")
fingerprint = hashlib.sha256(canonical(session)).hexdigest()
sizes = {
    method: {"compressed_bytes": 0, "uncompressed_bytes": 0}
    for method in ("C", "D")
}

for bit in range(8):
    record = read_record(FOLDER / f"bit_{bit}_result.json")
    if record["bit"] != bit or record["session_fingerprint"] != fingerprint:
        raise RuntimeError("Model session mismatch.")

    content = (FOLDER / f"bit_{bit}_models.joblib").read_bytes()
    if hashlib.sha256(content).hexdigest() != record["models_sha256"]:
        raise RuntimeError("Model checksum mismatch.")

    # Load only verified, locally produced artifacts.
    models = joblib.load(io.BytesIO(content))
    for method in ("C", "D"):
        for compression, key in (
            (3, "compressed_bytes"),
            (0, "uncompressed_bytes"),
        ):
            buffer = io.BytesIO()
            joblib.dump(models[method], buffer, compress=compression)
            sizes[method][key] += buffer.tell()

print("Eight predictors per method:")
print(json.dumps(sizes, indent=2))
print("\nSerialization sizes are not live RAM measurements.")
print("Thresholds, feature construction, metadata and scan costs are excluded.")
print("No predictions, fitting, or file writes.")

Eight predictors per method:
{
  "C": {
    "compressed_bytes": 582653,
    "uncompressed_bytes": 1618336
  },
  "D": {
    "compressed_bytes": 490424,
    "uncompressed_bytes": 1521824
  }
}

Serialization sizes are not live RAM measurements.
Thresholds, feature construction, metadata and scan costs are excluded.
No predictions, fitting, or file writes.


In [10]:
from pathlib import Path
import hashlib
import json

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
SOURCE = ROOT / "results/notebook07a/invariant_masks_v1"

def canonical(value):
    return json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        allow_nan=False, separators=(",", ":")
    ).encode("utf-8")

def verified_record(name, expected):
    content = (SOURCE / name).read_bytes()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError(f"File checksum mismatch: {name}")
    envelope = json.loads(content)
    payload = envelope["payload"]
    if hashlib.sha256(canonical(payload)).hexdigest() != envelope["payload_sha256"]:
        raise RuntimeError(f"Payload mismatch: {name}")
    return payload

hashes = {
    "manifest.json":
        "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a",
    "correctness_v1.json":
        "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206",
    "comparison_on_06b_targets_v1.json":
        "3410026a67e06483f4fff3e6823c9a7dcf47ee6c9c6d658693937983214e9254",
}
records = {
    name: verified_record(name, expected)
    for name, expected in hashes.items()
}

if set(sizes) != {"C", "D"}:
    raise RuntimeError("Serialization measurements are unavailable.")
for method in ("C", "D"):
    for key in ("compressed_bytes", "uncompressed_bytes"):
        if type(sizes[method][key]) is not int or sizes[method][key] <= 0:
            raise RuntimeError("Invalid serialization measurement.")

manifest = records["manifest.json"]
comparison = records["comparison_on_06b_targets_v1.json"]

report = {
    "status": "07A_FUNCTIONAL_BASELINE_COMPLETE_COST_AND_AP_STUDIES_PENDING",
    "source_file_sha256": hashes,
    "protocol_sha256": manifest["protocol_sha256"],
    "invariant_baseline_totals": manifest["totals"],
    "functional_checks": {
        key: records["correctness_v1.json"][key]
        for key in (
            "status",
            "clean_weights_checked_unchanged",
            "position_bit_cases_checked",
            "protected_cases_exactly_recovered",
            "unprotected_cases_left_unchanged",
        )
    },
    "common_test_targets": comparison["targets"],
    "common_target_methods": comparison["methods"],
    "predictor_serialization_bytes": sizes,
    "interpretation": [
        "C has higher exact recovery on the common inspected targets.",
        "The invariant baseline leaves clean weights unchanged under trusted masks.",
        "The baseline targets bits 4-6; C and D inspect all eight bits.",
        "Masks use clean reference weights, including test-channel weights.",
        "Predictor training uses the separate training-channel partition.",
        "Storage budgets and information assumptions differ.",
        "Compressed storage, uncompressed serialization and live RAM are distinct.",
        "No overall winner, independent confirmation or AP benefit is established."
    ],
    "pending": [
        "Full scan and correction timing.",
        "Live memory and feature-construction costs.",
        "A separate frozen protocol for equal-budget comparison.",
        "End-to-end evaluation of the invariant baseline.",
        "Git review and publication of new session files."
    ]
}

envelope = {
    "payload_sha256": hashlib.sha256(canonical(report)).hexdigest(),
    "payload": report,
}
content = (
    json.dumps(envelope, indent=2, allow_nan=False) + "\n"
).encode("utf-8")

destination = ROOT / "docs/research_reports/notebook07a_baseline_summary_v1.json"
destination.parent.mkdir(parents=True, exist_ok=True)

if destination.exists():
    if destination.read_bytes() != content:
        raise RuntimeError("Existing report differs; nothing overwritten.")
    print("IDENTICAL REPORT ALREADY EXISTS")
else:
    with destination.open("xb") as stream:
        stream.write(content)
    print("NEW REPORT SAVED")

assert destination.read_bytes() == content
print("Path:", destination)
print("SHA256:", hashlib.sha256(content).hexdigest())
print("No experiments, model changes, commit, or push.")

NEW REPORT SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/docs/research_reports/notebook07a_baseline_summary_v1.json
SHA256: 7cb1b0a831ed44ebbac3e3f91a7e6d64b963f427ab9fa479e96608be7348d10d
No experiments, model changes, commit, or push.


In [11]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_timing_v1.py"

source = r'''"""CPU timing of complete NumPy mask scans; not inference latency."""
import hashlib
import importlib.util
import io
import json
import platform
import time
from pathlib import Path

MANIFEST_SHA = "bee8dc2d17c1612ba1eddaeab8960f5d265e73c75c907c5cb80c62b0bb10f08a"
CHECK_SHA = "b0ec6384fdf6a5123710a4806f5848cf8cc93d72bf2f35c5ea0aab2f5aa7c206"
WARMUPS = 3
REPEATS = 20

def sha(content):
    return hashlib.sha256(content).hexdigest()

def canonical(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False,
                      allow_nan=False, separators=(",", ":")).encode()

def verified(path, expected):
    content = path.read_bytes()
    if sha(content) != expected:
        raise RuntimeError(f"Checksum mismatch: {path.name}")
    return content

def unpack(content):
    envelope = json.loads(content)
    if sha(canonical(envelope["payload"])) != envelope["payload_sha256"]:
        raise RuntimeError("Payload checksum mismatch.")
    return envelope["payload"]

def load_module(name, path, expected):
    verified(path, expected)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

def run(repo):
    import numpy as np
    import torch
    from importlib.metadata import version
    from threadpoolctl import threadpool_limits

    repo = Path(repo)
    folder = repo / "results/notebook07a/invariant_masks_v1"
    destination = folder / "cpu_scan_timing_v1.json"
    if destination.exists():
        raise FileExistsError("Timing report exists; inspect before rerunning.")
    manifest = unpack(verified(folder / "manifest.json", MANIFEST_SHA))
    checks = unpack(verified(folder / "correctness_v1.json", CHECK_SHA))
    checker = load_module(
        "verified07a_corrector", repo / "src/invariant07a_check_v1.py",
        checks["module_sha256"]
    )
    screen = load_module(
        "verified07a_timing_screen", repo / "src/screen05b_campaign_v1.py",
        manifest["source_screen_sha256"]
    )
    torch.set_num_threads(1)
    prepared = screen.Campaign(repo.parent)
    prepared._load_sources()
    prepared._model("cpu")
    if prepared.clean_hash != manifest["clean_model_state_sha256"]:
        raise RuntimeError("Clean state mismatch.")

    raw = verified(folder / "masks.npz", manifest["mask_archive_sha256"])
    clean, masks = [], []
    with np.load(io.BytesIO(raw), allow_pickle=False) as archive:
        for item in manifest["inventory"]:
            q = prepared.quantized[item["layer"]][0].cpu().contiguous().numpy().copy()
            packed = archive[item["array_key"]].copy()
            if sha(q.tobytes()) != item["clean_integer_array_sha256"]:
                raise RuntimeError("Clean tensor mismatch.")
            if sha(packed.tobytes()) != item["mask_array_sha256"]:
                raise RuntimeError("Mask mismatch.")
            clean.append(q)
            masks.append(packed)

    faulty = list(clean)
    selected_fault = None
    for layer_index, (q, packed) in enumerate(zip(clean, masks)):
        protected = (packed & 7) | ((packed >> 3) & 7)
        available = np.flatnonzero(protected)
        if not len(available):
            continue
        group = int(available[0])
        offset = next(b for b in range(3) if int(protected[group]) & (1 << b))
        bit = offset + 4
        o, i, kh, kw = q.shape
        if (kh, kw) == (3, 3):
            flat = group * 9
        else:
            per_channel = i // 9
            flat = (group // per_channel) * i + (group % per_channel) * 9
        faulty[layer_index] = q.copy()
        faulty[layer_index].view(np.uint8).reshape(-1)[flat] ^= np.uint8(1 << bit)
        selected_fault = {
            "layer": manifest["inventory"][layer_index]["layer"],
            "flat_index": flat, "bit": bit,
            "selection": "first_protected_position_to_exercise_correction_only"
        }
        break
    if selected_fault is None:
        raise RuntimeError("No protected fault available.")

    def scan(arrays):
        return [checker.correct(q, mask) for q, mask in zip(arrays, masks)]

    results = {}
    with threadpool_limits(limits=1):
        for label, arrays in (("clean_scan", clean), ("one_protected_fault_scan", faulty)):
            for _ in range(WARMUPS):
                output = scan(arrays)
                del output
            times = []
            for _ in range(REPEATS):
                started = time.perf_counter()
                output = scan(arrays)
                times.append(time.perf_counter() - started)
                # Validation and output deletion are outside the timed interval.
                if not all(np.array_equal(a, b) for a, b in zip(output, clean)):
                    raise RuntimeError("Scan output differs from clean reference.")
                del output
            results[label] = {
                "seconds": times,
                "median_seconds": float(np.median(times)),
                "min_seconds": float(np.min(times)),
                "empirical_p95_seconds": float(np.quantile(times, 0.95))
            }

    prepared._assert_state(prepared.clean_hash)
    payload = {
        "status": "CPU_NUMPY_SCAN_TIMING_COMPLETE_NOT_INFERENCE_OVERHEAD",
        "manifest_sha256": MANIFEST_SHA,
        "correctness_file_sha256": CHECK_SHA,
        "module_sha256": sha(Path(__file__).read_bytes()),
        "warmups": WARMUPS, "repeats": REPEATS,
        "order": ["clean_scan", "one_protected_fault_scan"],
        "weights_scanned_per_repeat": sum(q.size for q in clean),
        "input_array_bytes": sum(q.nbytes for q in clean),
        "mask_array_bytes": sum(m.nbytes for m in masks),
        "output_array_bytes_per_scan": sum(q.nbytes for q in clean),
        "selected_fault": selected_fault,
        "environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "numpy": version("numpy"), "torch": version("torch"),
            "threadpool_limit": 1
        },
        "results": results,
        "limitations": [
            "In-memory CPU NumPy implementation; outputs allocate new arrays.",
            "Model preparation, file reads, validation and output deletion excluded.",
            "Temporary allocation peaks and live RAM not measured.",
            "Warm-cache repeated measurements; not deployment latency guarantees.",
            "Selected protected fault is a functional timing workload, not an efficacy sample.",
            "No measured impact on complete YOLO inference latency."
        ]
    }
    content = (json.dumps({
        "payload_sha256": sha(canonical(payload)), "payload": payload
    }, indent=2, allow_nan=False) + "\n").encode()
    with destination.open("xb") as stream:
        stream.write(content)
    if destination.read_bytes() != content:
        raise RuntimeError("Write verification failed.")

    for label, result in results.items():
        print(
            f"{label}: median={result['median_seconds']:.6f}s | "
            f"empirical p95={result['empirical_p95_seconds']:.6f}s"
        )
    print("Weights per scan:", payload["weights_scanned_per_repeat"])
    print("Selected timing fault:", selected_fault)
    print("Saved:", destination)
    print("SHA256:", sha(content))
    return payload
'''

compile(source, str(path), "exec")
content = source.encode("utf-8")
if path.exists():
    if path.read_bytes() != content:
        raise RuntimeError("Existing module differs; nothing overwritten.")
    print("IDENTICAL MODULE ALREADY EXISTS")
else:
    with path.open("xb") as stream:
        stream.write(content)
    print("NEW MODULE SAVED")

assert path.read_bytes() == content
print("Path:", path)
print("Syntax: PASS")
print("Timing has not been executed.")

NEW MODULE SAVED
Path: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/src/invariant07a_timing_v1.py
Syntax: PASS
Timing has not been executed.


In [12]:
from pathlib import Path
import importlib.util

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")
path = ROOT / "src/invariant07a_timing_v1.py"

spec = importlib.util.spec_from_file_location("invariant07a_timing_v1", path)
timing07a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(timing07a)

timing_result07a = timing07a.run(ROOT)

clean_scan: median=0.044503s | empirical p95=0.055701s
one_protected_fault_scan: median=0.046084s | empirical p95=0.050897s
Weights per scan: 11146080
Selected timing fault: {'layer': 'model.0.conv', 'flat_index': 36, 'bit': 6, 'selection': 'first_protected_position_to_exercise_correction_only'}
Saved: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json
SHA256: 6972ab2c57f074358dd506a6cd79e8e9497658ed29fd7fbb88d60a55455deda9


In [14]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

commands = [
    ["git", "rev-parse", "HEAD"],
    ["git", "--no-optional-locks", "status", "--short",
     "--untracked-files=normal"],
    ["git", "diff", "--stat"],
]

for command in commands:
    print("\nCOMMAND:", " ".join(command))
    try:
        result = subprocess.run(
            command,
            cwd=ROOT,
            capture_output=True,
            text=True,
            timeout=120,
        )
        print("Return code:", result.returncode)
        print(result.stdout.strip() or "(no output)")
        if result.stderr.strip():
            print("STDERR:", result.stderr.strip())
        if result.returncode != 0:
            break
    except subprocess.TimeoutExpired:
        print("TIMEOUT: command exceeded 120 seconds; nothing changed.")
        break

print("\nRead-only inspection; no staging, commit, or push.")


COMMAND: git rev-parse HEAD
Return code: 0
97aca930e3c9fdb47e714e9d49decf4778d894c0

COMMAND: git --no-optional-locks status --short --untracked-files=normal
Return code: 0
?? docs/research_reports/notebook05b_screening_v1/progress_01600.json
?? docs/research_reports/notebook05b_screening_v1/progress_01604.json
?? docs/research_reports/notebook05b_screening_v1/progress_01650.json
?? docs/research_reports/notebook05b_screening_v1/progress_01700.json
?? docs/research_reports/notebook05b_screening_v1/progress_01704.json
?? docs/research_reports/notebook06b_end_to_end_v1/
?? docs/research_reports/notebook06b_per_layer_v1.csv
?? docs/research_reports/notebook06b_pilot_v1.md
?? docs/research_reports/notebook07a_baseline_summary_v1.json
?? experiments/notebook06b/
?? experiments/notebook07a/
?? src/calibration06b_v1.py
?? src/decisions06b_e2e_v1.py
?? src/end_to_end06b_v1.py
?? src/evaluation06b_v1.py
?? src/invariant07a_check_v1.py
?? src/invariant07a_compare_v1.py
?? src/invariant07a_extra

In [15]:
from pathlib import Path
import subprocess

ROOT = Path("/content/drive/MyDrive/YOLOv8_Fault_Tolerance/code")

print("Ignore rules for representative result files:")
paths = [
    "results/notebook06b/calibration_v1.json",
    "results/notebook06b/evaluation_v1/summary.json",
    "results/notebook06b/end_to_end_pilot_v1/session.json.gz",
    "results/notebook07a/invariant_masks_v1/manifest.json",
    "results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json",
]
result = subprocess.run(
    ["git", "check-ignore", "-v", "--", *paths],
    cwd=ROOT, capture_output=True, text=True, timeout=30,
)
print(result.stdout.strip() or "(no matching ignore rules)")
if result.stderr.strip():
    print("STDERR:", result.stderr.strip())
print("Return code:", result.returncode)

print("\nSaved notebook candidates:")
directories = [
    ROOT / "notebooks",
    Path("/content/drive/MyDrive/Colab Notebooks"),
]
for directory in directories:
    print("\nDirectory:", directory)
    if not directory.is_dir():
        print("NOT FOUND")
        continue
    matches = [
        path for path in directory.glob("*.ipynb")
        if path.name.startswith(("06b", "07a"))
    ]
    for path in sorted(matches):
        print(path.name, "| bytes:", path.stat().st_size)
    if not matches:
        print("NO MATCHES")

print("\nNew protocol/report files:")
for relative in (
    "experiments/notebook06b",
    "experiments/notebook07a",
    "docs/research_reports/notebook06b_end_to_end_v1",
):
    directory = ROOT / relative
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            print(path.relative_to(ROOT), "| bytes:", path.stat().st_size)

print("\nRead-only; no copying, staging, commit, or push.")

Ignore rules for representative result files:
.gitignore:24:results/*	results/notebook06b/calibration_v1.json
.gitignore:24:results/*	results/notebook06b/evaluation_v1/summary.json
.gitignore:24:results/*	results/notebook06b/end_to_end_pilot_v1/session.json.gz
.gitignore:24:results/*	results/notebook07a/invariant_masks_v1/manifest.json
.gitignore:24:results/*	results/notebook07a/invariant_masks_v1/cpu_scan_timing_v1.json
Return code: 0

Saved notebook candidates:

Directory: /content/drive/MyDrive/YOLOv8_Fault_Tolerance/code/notebooks
NO MATCHES

Directory: /content/drive/MyDrive/Colab Notebooks
06b_detection_repair_pilot.ipynb | bytes: 284647
07a_invariant_bit_protectionfrom google.colab import drive from pathlib import Path import hashlib import json import math  ~.mount(" content drive")  ROOT = Path(" content drive MyDrive YOLOv8_Fault_Tolerance code") path = ROOT   "experiments noteb.ipynb | bytes: 72128

New protocol/report files:
experiments/notebook06b/detection_repair_pilot_pr